In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from package.databases.initialize import initialize_memories

initialize_memories()

🔁 Trigger already exists, skipping.


In [17]:
from dataclasses import dataclass, field
from typing import List, Optional
import pickle
from package.flows.offline.flow import get_parallel_actions, Shared
from package.interface import SourceOptions
from package.utils.data_loder import PDFLoader
from package.databases.session import get_session, Depends, Session
from package.databases.management.document import DocumentManagement, Document
from package.databases.management.longterm import LongTermManagement, LongTerm

def create_document(source_path:str):
    dm = DocumentManagement()
    ltm = LongTermManagement()
    source_type = source_path.split(".")[-1]

    source_ops = SourceOptions(
        path=source_path,
        type=source_type if source_type in ['pdf'] else "other"
    )
    document = Document(source=source_path, type=source_type)
    document = dm.create_document(document, session=Depends(get_session))

    contexts = PDFLoader(source=source_ops).run()
    longterms = [LongTerm(document_id=document.id, raw=context.context, meta=context.metadata) for context in contexts]
    ltm.create_raws(longterms, session=Depends(get_session))

    _longterms = ltm.read_longterms_by_document(document_id=document.id, session=Depends(get_session))
    longterms = []
    for l in _longterms:
        _l = l.__dict__.copy()
        if hasattr(_l, '_sa_instance_state'):
            delattr(_l, '_sa_instance_state')
        longterms.append(LongTerm(**_l))

    return document, longterms

@dataclass
class OfflineIndexRunner:
    content_path: str
    enrich_system_prompt: str = './package/flows/offline/prompts/enricher.md'
    term_system_prompt: str = './package/flows/offline/prompts/term_extractor.md'
    term_model:str = 'us.meta.llama4-maverick-17b-instruct-v1:0'
    chunks: List[LongTerm] = field(default_factory=list)
    current_chunk_index: int = 0
    completed_count: int = 0
    state_path: str = './offline_index_state.pkl'
    document_id: Optional[str] = None
    document_source: Optional[str] = None
    
    def load_content(self):
        document, longterms = create_document(self.content_path)
        self.document_id = document.id
        self.document_source = document.source
        self.chunks.extend(longterms)

    def run(self):
        if len(self.chunks)==0:
            self.load_content()
        try:
            for i in range(self.current_chunk_index, len(self.chunks)):
                self.current_chunk_index = i
                chunk = self.chunks[i]
                
                # Process single chunk: Parallel(Enrich, Jargon) + Embed
                shared = Shared(
                    enrich_system_prompt=self.enrich_system_prompt,
                    term_system_prompt=self.term_system_prompt,
                    term_model=self.term_model,
                    chunk=chunk
                )
                flow = get_parallel_actions()
                flow.run(shared)
                self.completed_count += 1
                
        except KeyboardInterrupt:
            print("Interrupted by user. Saving state...")
            self.save_state()
            print(f"State saved. Completed {self.completed_count} tasks.")
            raise                
        except Exception as e:
            print("Error:", str(e))
            self.save_state()
            raise  

    def save_state(self):
        with open(self.state_path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load_state(cls):
        try:
            with open(cls.state_path, 'rb') as f:
                return pickle.load(f)
        except FileNotFoundError as e:
            print("Error:", str(e))
            return None

In [4]:
# runner = OfflineIndexRunner(
#     content_path='./sources/Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf',
# )
# runner.run()

In [5]:
from pathlib import Path
from tqdm import tqdm

source_dir = Path("./sources")
pdf_files = source_dir.glob("*.pdf")

In [6]:
# exists = ["Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf"]
exists = []
errors = []
for pdf_path in tqdm(pdf_files):
    source = pdf_path.name
    state_file = source.split(".pdf")[0].lower().replace("-",'_').replace(" ", "_")
    if source in exists:
        continue
    print(source)
    try:
        runner = OfflineIndexRunner(
            content_path=str(pdf_path),
            state_path="./experiments/offline/state/v1/{filename}".format(filename=state_file)
        )
        runner.run()
    except Exception as e:
        errors.append(dict(
            filename=source,
            statename=state_file,
            error=str(e)
        ))
        print("ERROR: {source}".format(source=source))

0it [00:00, ?it/s]

AcuRank Uncertainty-Aware Adaptive Computation for Listwise Reranking.pdf


d:\broai-arai\backend\package\utils\data_loder.py:22: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: split_markdown
  chunks = split_markdown(text)
d:\broai-arai\backend\package\utils\data_loder.py:23: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: consolidate_markdown
  consolidated_chunks = consolidate_markdown(chunks)
d:\broai-arai\backend\package\utils\data_loder.py:24: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: get_markdown_sections
  sections = get_markdown_sections(consolidated_chunks)
d:\broai-arai\backend\package\utils\data_loder.py:30: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: split_overlap
  new_contexts = split_overlap(contexts, max_tokens=max_tokens, overlap=overlap)


Markdown headings: max(2)


0it [08:53, ?it/s]

Interrupted by user. Saving state...
State saved. Completed 182 tasks.


KeyboardInterrupt: 

In [3]:
from package.databases.management.longterm import LongTermManagement, LongTerm
from package.databases.management.term import TermManagement, Term
from package.databases.management.document import DocumentManagement, Document
from package.databases.session import Depends, get_session

In [4]:
tm = TermManagement()
dm = DocumentManagement()
ltm = LongTermManagement()

In [5]:
documents = dm.read_documents(session=Depends(get_session))
# for document in documents:
#     document.source = document.source.split("\\")[-1]
documents

[Document(source='AcuRank Uncertainty-Aware Adaptive Computation for Listwise Reranking.pdf', type='pdf', created_at=datetime.datetime(2025, 7, 4, 13, 56, 29, 16229), id='af617836-a979-448c-bfcf-7d6385fd12a1', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 7, 4, 13, 56, 29, 16229)),
 Document(source='Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf', type='pdf', created_at=datetime.datetime(2025, 7, 4, 13, 58, 1, 676670), id='06ad7bfc-20e0-40fb-a806-66c8f871e96a', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 7, 4, 13, 58, 1, 676670)),
 Document(source='ClueAnchor Clue-Anchored Knowledge Reasoning Exploration and Optimization for Retrieval-Augmented Generation.pdf', type='pdf', created_at=datetime.datetime(2025, 7, 4, 13, 59, 6, 52446), id='44316211-6f8d-463a-ab44-fad555690d5c', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 7, 4, 13, 59, 6, 52446)),


In [51]:
test = {"test":[1,2,3]}
len(list(test.values()))

1

In [78]:
len("   ".replace("  ", ""))

1

In [74]:
len(test.items())

1

In [73]:
source_term = 'freshwiki'
_terms = tm.read_similar_terms(source_term, session=Depends(get_session))
for term in _terms:
    mask = True
    mask &= source_term.lower() != term.evidence.lower().strip()
    mask &= term is not None    
    if mask:
        print(" | ".join([term.term, term.evidence, term.explanation]))

FreshWiki | FreshWiki, a dataset of recent high-quality Wikipedia articles | FreshWiki is a dataset of recent high-quality Wikipedia articles
FreshWiki | FreshWiki dataset (§2.1) | FreshWiki is a dataset curating recent, high-quality Wikipedia articles
FreshWiki dataset | randomly select 100 samples from the FreshWiki dataset | FreshWiki dataset is a dataset used for the experiment
FreshWiki | we curate the FreshWiki dataset | FreshWiki is a dataset of recent and high-quality English Wikipedia articles
FreshWiki | FreshWiki dataset | FreshWiki is a dataset curated for the study
FreshQA | publicly available datasets, including FreshQA | FreshQA is a publicly available dataset
FreshQA | FreshQA [57] | FreshQA is a dataset used for evaluation
FreshQA | FreshQA is a column header in the table | FreshQA is likely a dataset or benchmark name
FreshQA | FreshQA [57] to evaluate its performance on single-hop reasoning questions | FreshQA is a dataset used to evaluate performance on single-hop r

In [76]:
_terms[0]

Term(type='Proper Name', term='FreshWiki', explanation='FreshWiki is a dataset of recent high-quality Wikipedia articles', longterm_id='a4cba954-62e3-4d68-b0c0-d1eefe9a63d4', created_at=datetime.datetime(2025, 8, 6, 18, 40, 57, 338817), search_vector="'freshwiki':1", evidence='FreshWiki, a dataset of recent high-quality Wikipedia articles', id='cd29349e-c1f8-42e6-8c74-cf7738b3d1e4', document_id='8170f3b0-9203-4ec2-8dcc-476eeb9c69b0', meta={'type': 'pdf', 'source': 'sources\\Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf', 'section': 'Abstract', 'sequence': 2}, updated_at=datetime.datetime(2025, 8, 6, 18, 40, 57, 338817))

In [35]:
longterms = ltm.read_longterms(session=Depends(get_session))
for longterm in longterms:
    longterm.meta['source'] = longterm.meta['source'].split("\\")[-1]

In [37]:
ltm.update_longterms(longterms, session=Depends(get_session))

In [38]:
for document in documents:
    source = document.source
    longterms = ltm.read_longterms_by_document(document.id, session=Depends(get_session))
    break
    for longterm in longterms:
        longterm.metadata

In [39]:
document.source

'GainRAG Preference Alignment in Retrieval-Augmented Generation through Gain Signal Synthesis.pdf'

In [17]:
from package.embedding.baai import BAAIEmbedding
embedder = BAAIEmbedding()

d:\broai-arai\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔍 Loading model from: BAAI/bge-m3


Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


In [40]:
vector = embedder.run(sentences=["What is {term}".format(term=source)])[0]

d:\broai-arai\backend\.venv\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [41]:
# vector = []
ltm.read_similar_text_with_like_source(
    vector,
    embed_method='raw',
    session=Depends(get_session),
    sources=[source]
)

[LongTerm(raw='## GainRAG: Preference Alignment in Retrieval-Augmented Generation through Gain Signal Synthesis\n', raw_embedding=array([-0.0092392 , -0.02253723, -0.04473877, ...,  0.0383606 ,
         0.02418518, -0.01670837], shape=(1024,), dtype=float32), enrich_embedding=array([-0.01023865, -0.01986694, -0.01609802, ...,  0.01579285,
         0.02275085, -0.0164032 ], shape=(1024,), dtype=float32), combo_embedding=array([-0.00266457, -0.02226257, -0.02680969, ...,  0.00577164,
         0.01829529, -0.00852203], shape=(1024,), dtype=float32), document_id='281cfb03-6082-438e-acaf-b6c90fe48c2c', updated_at=datetime.datetime(2025, 8, 7, 14, 34, 28, 694788), id='467a8aed-d9cd-4c7a-b74b-28a546b52cb6', enrich='GainRAG is a method that aligns preferences in retrieval-augmented generation by synthesizing a gain signal. It aims to improve the quality of generated text by optimizing the retrieval process.', combo='GainRAG is a method that aligns preferences in retrieval-augmented generation 

In [14]:
longterms[0].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x223ff921f10>,
 'raw': "1 Introduction\n\nModern information retrieval pipelines, such as web search and retrieval-augmented generation (RAG) systems, typically adopt a fast first-phase retriever, selecting a set of broadly relevant documents, like BM25 [1] or dense encoders [2] optimized for recall and speed. As they often produce noisy or suboptimal rankings, reranking is critical in applications where precision at the top is essential, including conversational agents and reasoning with large language models (LLMs) [3, 4].\n\nReranking methods can be broadly categorized by how they model document interactions and handle relative relevance. Pointwise methods [5] independently assign scores to each document, offering scalability but failing to consider the competitive context among candidates. Pairwise methods [6] improve upon this by comparing document pairs to capture local preferences, but often struggle to maintain cohe

In [22]:
from package.databases.utils import now_utc
from package.embedding.baai import BAAIEmbedding
embedder = BAAIEmbedding()

def embed(longterms, embedder, session: Session = Depends(get_session)):
    updated_at = now_utc()
    raws = [longterm.raw for longterm in longterms]
    raw_vectors = embedder.run(sentences=raws)
    enrichs = [longterm.enrich for longterm in longterms]
    enrich_vectors = embedder.run(sentences=enrichs)
    combos = [longterm.combo for longterm in longterms]
    combo_vectors = embedder.run(sentences=combos)
    for longterm, vector in zip(longterms, raw_vectors):
        longterm.raw_embedding = vector
    for longterm, vector in zip(longterms, enrich_vectors):
        longterm.enrich_embedding = vector
    for longterm, vector in zip(longterms, combo_vectors):
        longterm.combo_embedding = vector
        longterm.updated_at = updated_at
    ltm.update_longterms(longterms=longterms, session=session)

🔍 Loading model from: BAAI/bge-m3


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 19834.35it/s]


In [23]:
for document in documents:
    longterms = ltm.read_longterms_by_document(document_id=document.id, session=Depends(get_session))
    embed(longterms, embedder=embedder, session=Depends(get_session))

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 37.22it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]


In [26]:
ltm.read_longterms_by_document(document.id, session=Depends(get_session))[0].__dict__

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState at 0x1d20b43c770>,
 'enrich': "TracLLM outperforms other methods in tracing back to malicious texts responsible for attacker-desired outputs, achieving higher precision and recall in various settings. Its effectiveness is particularly notable when an attacker injects a small number of malicious instructions/texts, where TracLLM's ability to consider the influence of each text when combined with others allows it to more effectively isolate and identify malicious content.",
 'id': 'a305a3d4-a349-460a-bac4-696d9df21a28',
 'combo': "TracLLM outperforms other methods in tracing back to malicious texts responsible for attacker-desired outputs, achieving higher precision and recall in various settings. Its effectiveness is particularly notable when an attacker injects a small number of malicious instructions/texts, where TracLLM's ability to consider the influence of each text when combined with others allows it to more effectively is

In [15]:
tm.read_terms(session=Depends(get_session))

[Term(evidence='AcuRank is a novel reranking framework', id='ce1a4a79-4d1d-4b04-9cd5-243d64bfafbd', document_id='d6077d6c-5ea2-41fb-b7e5-517ef9922722', meta={'type': 'pdf', 'source': 'sources\\AcuRank Uncertainty-Aware Adaptive Computation for Listwise Reranking.pdf', 'section': '## AcuRank: Uncertainty-Aware Adaptive Computation for Listwise Reranking', 'sequence': 0}, updated_at=datetime.datetime(2025, 8, 6, 18, 25, 17, 617513), term='AcuRank', type='Framework', explanation='AcuRank is a reranking framework that incorporates uncertainty-aware adaptive computation', longterm_id='60ca5897-6445-4332-a573-be823bc13ecd', created_at=datetime.datetime(2025, 8, 6, 18, 25, 17, 617513), search_vector="'acurank':1"),
 Term(evidence='listwise ranking loss function, which considers the entire ranked list when computing the loss', id='1e710820-dd6a-4413-a87f-dd91c6886eeb', document_id='d6077d6c-5ea2-41fb-b7e5-517ef9922722', meta={'type': 'pdf', 'source': 'sources\\AcuRank Uncertainty-Aware Adaptiv

In [15]:
target = "STORM"
detailed_terms = []
terms = tm.read_terms(session=Depends(get_session))
for i in terms:
    if target.lower() == i.term.lower():
        if target.lower() != i.evidence.lower().strip():
            print("{term} | {type} | {evidence} | {explanation}".format(term=i.term, type=i.type, evidence=i.evidence, explanation=i.explanation))
            print("="*20)
            detailed_terms.append("{evidence}".format(evidence=i.evidence))

STORM | Acronym | STORM , a writing system for the S ynthesis of T opic O utlines through R etrieval and M ulti-perspective Question Asking | STORM stands for Synthesis of Topic Outlines through Retrieval and Multi-perspective Question Asking
STORM | Acronym | we propose the STORM paradigm for the S ynthesis | STORM is a paradigm proposed for Synthesis
STORM | Acronym | STORM paradigm for the S ynthesis of T opic O utlines through R etrieval and M ulti-perspective Question Asking | STORM is a paradigm for the synthesis of topic outlines through retrieval and multi-perspective question asking
STORM | Technical Term | We present STORM to automate the pre-writing stage | STORM is a system that automates the pre-writing stage
STORM | Proper Name | STORM discovers different perspectives by surveying existing articles from similar topics | STORM is a system or method that discovers different perspectives on a topic
STORM | Proper Name | STORM simulates a conversation | STORM is a system that

In [39]:
len(detailed_terms)

21

In [29]:
from brollm import BedrockChat

model = BedrockChat()

In [30]:
BedrockChat()

In [35]:
from enum import Enum

class ModelName(Enum):
    marverick = "us.meta.llama4-maverick-17b-instruct-v1:0"
    llama32_11b = "us.meta.llama3-2-11b-instruct-v1:0"

In [23]:
dir(ModelName.marverick)

['__class__', '__doc__', '__eq__', '__hash__', '__module__', 'name', 'value']

In [40]:
with open("./package/flows/online/prompts/v1/chat.md",'r') as f:
    system_prompt = f.read()

user_input = "What is STORM?"

model.model_name = ModelName.llama32_11b.value
model.run(system_prompt=system_prompt, messages=[
    model.UserMessage(text="TERMS: \n\n{term}\n\nQUESTION: \n\n{user_input}\n\n".format(user_input=user_input, term=detailed_terms))
])

'Based on the provided TERMS, STORM is a writing system for the Synthesis of Topic Outlines through Retrieval and Multi-perspective Question Asking. It is an LLM-based writing system that automates the pre-writing stage by discovering different perspectives, simulating multi-turn conversations, and creating outlines.'

In [42]:
model.model_name = ModelName.marverick.value

model.run(system_prompt=system_prompt, messages=[
    model.UserMessage(text="TERMS: \n\n{term}\n\nQUESTION: \n\n{user_input}\n\n".format(user_input=user_input, term=detailed_terms))
])

'STORM is a writing system that automates the pre-writing stage by synthesizing topic outlines through retrieval and multi-perspective question asking. It is an LLM-based framework that discovers different perspectives by surveying existing articles, simulates conversations, and creates an outline. STORM is built using zero-shot prompting with models like gpt-3.5-turbo for question asking and gpt-3.5-turbo-instruct for other parts of the system.'

In [13]:
dm.read_documents(session=Depends(get_session))

[Document(source='./sources/Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf', type='pdf', created_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884), id='cf80c844-5632-4384-b75a-50e3f64f30a0', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884))]